# Priors and likelihoods

Calibration targets already merge observation times into a save plan. This page
adds the missing probability: named **priors**, **likelihoods** on each target,
a traceable `TargetSet.log_likelihood`, and a **`BayesianModel`** that turns
those pieces into MAP and MCMC.

The claims to check:

1. A `Uniform` prior's numpyro density peaks inside `[lo, hi]` and is zero outside.
2. A Normal likelihood on an SIR infecteds series scores the true trajectory
   higher than a deliberately wrong one.
3. A prior-valued `sd` (Kiribati-style hierarchical scale) is read from `params`
   and changes the log-likelihood.
4. `BayesianModel.find_map` recovers the infection rate that generated the data.

Full posterior scenario runs are the next roadmap step (`18-calibration.ipynb`).


In [ ]:
from typing import Any, NamedTuple

import jax.numpy as jnp
import numpy as np
import numpyro.distributions as dist
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)
from summer4.epi.calibration import (
    BayesianModel,
    NormalLikelihood,
    Uniform,
    priors_from_frame,
)


## Prior density

`Uniform("contact", 0.1, 0.5).to_numpyro()` is a numpyro distribution. Plot its
log-density on a grid that straddles the bounds: inside the interval the density
is flat and positive; outside it is `-inf`.


In [ ]:
prior = Uniform("contact", 0.1, 0.5)
d = prior.to_numpyro()
assert prior.bounds() == (0.1, 0.5)

grid = np.linspace(0.0, 0.6, 241)
log_p = np.asarray(d.log_prob(grid))
finite = np.isfinite(log_p)

frame = pd.DataFrame({"x": grid, "log_prob": np.where(finite, log_p, np.nan)})
figure = frame.plot(x="x", y="log_prob", title="Uniform(0.1, 0.5) log-density")
figure.update_layout(xaxis_title="contact rate", yaxis_title="log density")

assert finite[(grid >= 0.1) & (grid <= 0.5)].all()
assert (~finite[(grid < 0.1) | (grid > 0.5)]).all()
assert float(d.log_prob(0.3)) > float("-inf")


## Likelihood on a sparse SIR target

Fit nothing yet — just score. Build an SIR, observe `I` at a handful of times,
attach `NormalLikelihood(sd=...)`, and compare `log_likelihood` at the true
infection rate versus a wrong one. The true trajectory must win.


In [ ]:
class Rates(NamedTuple):
    infection: float
    recovery: float


state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))

true = Rates(infection=0.35, recovery=0.1)
wrong = Rates(infection=0.15, recovery=0.1)
obs_t = np.array([0.0, 10.0, 20.0, 40.0, 60.0, 80.0])
qty = Compartments(where=state["I"])

truth = cm.run(
    true,
    y0,
    t0=0.0,
    t1=80.0,
    dt=1.0,
    save=SavePlan(requests={"I": SaveRequest(qty, ts=obs_t)}),
    solver="euler",
)
raw = truth["I"].at_times(obs_t).values
obs = np.asarray(raw.data if hasattr(raw, "data") else raw).reshape(-1)

lik = NormalLikelihood.from_tolerance(obs, tol_pct=15.0)
targets = TargetSet(
    targets=(Target(key="I", times=obs_t, values=obs, quantity=qty, likelihood=lik),)
)
plan = targets.plan(SavePlan())

true_res = cm.run(true, y0, t0=0.0, t1=80.0, dt=1.0, save=plan, solver="euler")
wrong_res = cm.run(wrong, y0, t0=0.0, t1=80.0, dt=1.0, save=plan, solver="euler")

ll_true = float(targets.log_likelihood(true_res, {}))
ll_wrong = float(targets.log_likelihood(wrong_res, {}))
print(f"log_likelihood true={ll_true:.3f}, wrong={ll_wrong:.3f}, sd={lik.sd:.3f}")
assert ll_true > ll_wrong


In [ ]:
def series_at(result: Any, times: np.ndarray) -> np.ndarray:
    raw = result["I"].at_times(times).values
    return np.asarray(raw.data if hasattr(raw, "data") else raw).reshape(-1)


compare = pd.DataFrame(
    {
        "truth / observations": obs,
        "true params": series_at(true_res, obs_t),
        "wrong params": series_at(wrong_res, obs_t),
    },
    index=obs_t,
)
figure = compare.plot(title="Infecteds at observation times")
figure.update_layout(xaxis_title="time (days)", yaxis_title="people")

np.testing.assert_allclose(series_at(true_res, obs_t), obs, rtol=1e-5)


## Hierarchical target scale

Kiribati puts a prior on the observation standard deviation itself
(`mixing_dist_sd ~ Uniform(5, 20)`). Pass that prior as `NormalLikelihood(sd=...)`;
`log_likelihood` looks the name up in `params`. A tighter sd must score a perfect
match higher than a looser one.


In [ ]:
sd_prior = Uniform("obs_sd", 5.0, 20.0)
hier = TargetSet(
    targets=(
        Target(
            key="I",
            times=obs_t,
            values=obs,
            quantity=qty,
            likelihood=NormalLikelihood(sd=sd_prior),
        ),
    )
)

ll_tight = float(hier.log_likelihood(true_res, {"obs_sd": 5.0}))
ll_loose = float(hier.log_likelihood(true_res, {"obs_sd": 20.0}))
print(f"hierarchical sd=5 → {ll_tight:.3f}; sd=20 → {ll_loose:.3f}")
assert ll_tight > ll_loose

# Table builder used for Kiribati parameters.xlsx constant sheet.
built = priors_from_frame(
    {
        "name": ["obs_sd", "infection"],
        "dist": ["uniform", "normal"],
        "p1": [5.0, 0.3],
        "p2": [20.0, 0.1],
    },
    "name",
    "dist",
    "p1",
    "p2",
)
assert built[0].name == "obs_sd"
assert built[0].bounds() == (5.0, 20.0)


## MAP with `BayesianModel`

Wrap the same SIR targets in a `BayesianModel` with a `Uniform` prior on the
infection rate. `find_map` maximises the joint density with optax and must land
near the true infection rate. Run-start transforms belong on
`FlowModel.compile(prepare_fn=...)`, not a separate `preprocess=` hook.


In [ ]:
bm = BayesianModel(
    cm,
    {"infection": 0.2, "recovery": 0.1},
    priors=(Uniform("infection", 0.05, 0.8),),
    targets=targets,
    y0=y0,
    run_kwargs=dict(t0=0.0, t1=80.0, dt=1.0, solver="euler"),
)
mapped = bm.find_map(steps=100, seed=0)
infection_hat = float(mapped["infection"])
print(f"MAP infection={infection_hat:.3f} (truth {true.infection})")

frame = pd.DataFrame(
    {
        "value": [true.infection, infection_hat],
    },
    index=["truth", "MAP"],
)
figure = frame.plot.bar(title="Infection rate: truth vs MAP")
figure.update_layout(yaxis_title="infection rate", showlegend=False)

assert abs(infection_hat / true.infection - 1.0) < 0.1
